In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import PeftModel
import os

# Cell 1: Define Custom PEFT Model Class (No changes needed, this is correct)
# This class correctly handles passing 'input_features' for Whisper models.
from peft import PeftModelForSeq2SeqLM

class WhisperPeftModel(PeftModelForSeq2SeqLM):
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        inputs_embeds=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        decoder_inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        task_ids=None,
        **kwargs
    ):
        # For Whisper, we expect 'input_features' in kwargs instead of 'input_ids'
        input_features = kwargs.pop("input_features", None)

        forward_kwargs = {}
        if input_features is not None:
            forward_kwargs["input_features"] = input_features
        if attention_mask is not None:
            forward_kwargs["attention_mask"] = attention_mask
        if inputs_embeds is not None:
            forward_kwargs["inputs_embeds"] = inputs_embeds
        if decoder_input_ids is not None:
            forward_kwargs["decoder_input_ids"] = decoder_input_ids
        if decoder_attention_mask is not None:
            forward_kwargs["decoder_attention_mask"] = decoder_attention_mask
        if decoder_inputs_embeds is not None:
            forward_kwargs["decoder_inputs_embeds"] = decoder_inputs_embeds
        if labels is not None:
            forward_kwargs["labels"] = labels
        if output_attentions is not None:
            forward_kwargs["output_attentions"] = output_attentions
        if output_hidden_states is not None:
            forward_kwargs["output_hidden_states"] = output_hidden_states
        if return_dict is not None:
            forward_kwargs["return_dict"] = return_dict

        # Pass any remaining kwargs
        forward_kwargs.update(kwargs)

        # Call the base model with adjusted kwargs
        return self.base_model(**forward_kwargs)

print("✅ Custom PEFT Model class defined!")

# Define file paths
merged_model_path = "/data/whisper/train/medium/102-sergical/merged_model"
checkpoint_path = "/data/whisper/train/medium/301-Mx01/checkpoint-8840"

# Cell 2: Load the base model with the CORRECT data type
print("Loading base model...")
# Load the base model using bfloat16 to match your training environment.
# This prevents numerical precision and range issues.
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    merged_model_path,
    torch_dtype=torch.bfloat16, ### CORRECTED ###
    device_map="auto",
)

# Load the processor (it has no dtype)
processor = AutoProcessor.from_pretrained(merged_model_path)

print("✅ Base model loaded!")

# Cell 3: Load the new LoRA adapter
print(f"\nLoading LoRA adapter from {checkpoint_path}...")

# Load the PEFT model. It will automatically use the bfloat16 dtype from the base_model.
peft_model = PeftModel.from_pretrained(
    base_model, 
    checkpoint_path,
    device_map="auto",
)

# Set the custom class to handle Whisper's specific inputs
peft_model.__class__ = WhisperPeftModel

print("✅ LoRA adapter loaded!")
peft_model.print_trainable_parameters()

# Cell 4: Merge the LoRA weights into the base model
print("\nMerging LoRA weights into base model...")

# The merge operation will now happen in bfloat16, preserving your trained weights correctly.
merged_model = peft_model.merge_and_unload()

print("✅ Model merged successfully!")

# Cell 5: Save the fully merged model
# Using a slightly different name to indicate the dtype, which is good practice.
final_output_dir = "/data/whisper/train/medium/301-Mx01/merged_model_bf16"
os.makedirs(final_output_dir, exist_ok=True)

print(f"\nSaving merged model to {final_output_dir}...")

# Save the final, merged model
merged_model.save_pretrained(final_output_dir)

# Also save the processor so the model is self-contained
processor.save_pretrained(final_output_dir)

print(f"✅ Merged model saved to: {final_output_dir}")

# Cell 6: Verify the saved model (Optional but recommended)
print("\nVerifying saved model...")

# Load the final model from the new directory, again using the correct dtype.
test_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    final_output_dir,
    torch_dtype=torch.bfloat16, ### CORRECTED ###
    device_map="auto",
)

print("✅ Model verified successfully!")
print(f"\nModel size: {test_model.num_parameters():,} parameters")

# Display the model structure to confirm everything looks right
print("\nModel structure:")
print(test_model)


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Custom PEFT Model class defined!
Loading merged base model...
✅ Base merged model loaded!

Loading LoRA adapter from /data/whisper/train/medium/102-sergical/continued_training/checkpoint-22988/...
✅ LoRA adapter loaded!
trainable params: 0 || all params: 798,460,928 || trainable%: 0.0000

Merging LoRA weights into base model...
✅ Model merged successfully!

Saving merged model to /data/whisper/train/medium/102-sergical/final_merged_model...
✅ Merged model saved to: /data/whisper/train/medium/102-sergical/final_merged_model

Verifying saved model...
✅ Model verified successfully!

Model size: 763,857,920 parameters

Model structure:
WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLay